# radcoolpv — radiative cooling of silicon PV

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/radcoolpv-py/blob/main/docs/site/notebooks/radcoolpv_colab.ipynb)

Edit the YAML, run the cell, read the numbers. Nothing is installed on your own
machine and nothing here needs a compiled electromagnetic solver.

**Run all** takes about a minute and produces the **temperatures** a module
settles at, the **powers** that balance there, the full set of **PV
parameters**, and the figures.

The three validation notebooks reproduce one published paper group by group:
[optics](validation_a_optics.ipynb), [cooling](validation_b_cooling.ipynb),
and [PV](validation_c_pv.ipynb).

## Set up the runtime

Colab runtimes are temporary. Run this again after a reset.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

PROJECT = Path("/content/radcoolpv-py")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/gsilvaoelker/radcoolpv-py.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", "."],
               cwd=PROJECT, check=True)
os.chdir(PROJECT)

from radcoolpv import config, pipeline, report
print("radcoolpv ready in", PROJECT)

## PV parameters, with no solver

Optics, energy balance and the single-diode cell, end to end, from a committed
free-form structure. Every quantity below is a result of this run.

In [ ]:
CASE = "examples/freeform_pv.yaml"
ctx = pipeline.run(config.load_cases(CASE)[0])
report.summary(ctx)

## Edit this and run your own case

Everything below is yours: the wavelength range, the layer thicknesses, the
materials, the geometry, the ambient temperature, the convection coefficient.

Three things worth knowing:

* **The wavelength range must fit inside every material's table.** The error
  names the file that is too narrow. `RII_Olmon_2012_ev_Au` stops at 24.93 µm.
* **Commenting a key out reverts it to the code default, which is often the
  opposite of what you wanted.** `# thermal: false` turns thermal *on*. Set
  values explicitly. Commenting out a list item, such as a `structure` layer,
  is safe.
* **`n` is not a free parameter.** Every spectral integral is trapezoidal on
  your grid, so the grid has to resolve the bands that matter.

In [ ]:
%%writefile my_case.yaml
# Cooling curve for a surface whose emittance you supply.
run:
  optics: false               # no solver: read the spectrum from a file
  thermal: true
  plots: true
  mode: cooling_curve
  write_outputs: true
  results_dir: results/my_case
  optics_results: validation/data/fig5a_measured_emittance.txt
  optics_results_angles: hemispherical
  optics_results_emittance_column: 3     # 1 bare, 2 flat silica, 3 cylinders

simulation:
  wavelength: {min: 2.0, max: 16.0, n: 281}
  angles: hemispherical

thermal:
  ambient_temperature: 300.0       # K
  convection_coefficient: 12.54    # W/m2-K, everything non-radiative
  absorbed_solar_power: 808.0      # W/m2 absorbed, not incident irradiance
  equilibrium: auto
  cooling_temperature: {min: 260.0, max: 380.0, n: 121}

In [ ]:
CASE = "my_case.yaml"
ctx = pipeline.run(config.load_cases(CASE)[0])
report.summary(ctx)

## Use your own spectrum

This case reads its optics from a file rather than computing them, so what it
needs from you is a **spectrum**: wavelength in micrometres in the first column,
one or more emittance columns after it. Upload it, then set `optics_results` to
the filename and `optics_results_emittance_column` to the column you want.

**If your spectrum reaches below about 1.1 µm you also get the PV parameters.**
Above the band gap essentially everything absorbed is absorbed in the silicon,
so radcoolpv takes the silicon absorptance to equal the emittance there and zero
below; `run.json` records that this was assumed rather than solved.

Uploading a material would do nothing here: with `optics: false` no material
table is ever consulted. To change materials, compute the optics from a geometry
instead — that is what [Validation A](validation_a_optics.ipynb) does.


In [ ]:
from google.colab import files

for name, blob in files.upload().items():
    rows = [r for r in blob.decode(errors="ignore").splitlines()
            if r.strip() and not r.lstrip().startswith("#")]
    try:
        columns = len([float(v) for v in rows[0].split()])
    except (IndexError, ValueError):
        columns = 0
    if columns < 2:
        print(f"{name}: not usable. Expected whitespace-separated numbers, "
              f"wavelength_um first and at least one spectrum after it "
              f"('#' comments allowed).")
    else:
        print(f"{name}: {len(rows)} rows, {columns} columns. Set optics_results "
              f"to {name!r} and optics_results_emittance_column to "
              f"1..{columns - 1}.")
